为了让你感受到 Semantic Kernel 在**企业级复杂场景**中的能力，我们来构建一个 **"智能 HR 入职与合规助手" (HR Compliance Agent)**。

### 这个案例复杂在哪里？

这个案例将展示 **RAG（检索增强生成）** 与 **Agent 工具链** 的深度结合。
普通的 Agent 只能查库（查 SQL），普通的 RAG 只能问答（读文档）。而这个 Agent 需要：

1.  **知识检索 (RAG)**：去查阅公司的《员工手册》和《合规章程》（非结构化文本）。
2.  **数据查询 (DB)**：去查员工数据库，看这个人的职级、入职时间、违规记录（结构化数据）。
3.  **逻辑推断 (Reasoning)**：结合 **手册规定** 和 **员工数据**，判断该员工是否有资格申请某项福利。
4.  **执行动作 (Action)**：如果通过，发批准邮件；如果不通过，发驳回通知。

---

### 完整代码实现

我们将模拟一个场景：**员工 "Alice" 申请 "远程办公 (WFH)"，AI 需要根据公司政策和她的个人状态自动审批。**

```python
import asyncio
import os
import dotenv
from typing import List

from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion, OpenAIChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function

# 解决 Jupyter/Notebook 环境报错
import nest_asyncio
nest_asyncio.apply()

dotenv.load_dotenv()

# ==============================================================================
# 1. 模拟向量数据库 (RAG 知识库)
# ==============================================================================
class CompanyPolicyPlugin:
    """
    负责检索公司非结构化的规章制度 (模拟 RAG)
    """
    def __init__(self):
        # 模拟存入向量库的文档切片
        self.knowledge_base = {
            "remote_work": "公司远程办公(WFH)政策：1. 员工必须入职满 6 个月。 2. 过去 30 天内没有迟到早退记录。 3. 实习生(Intern)无权申请。",
            "reimbursement": "报销政策：餐饮报销上限为 50 元/天，交通报销需要发票。",
            "security": "安全政策：所有员工必须完成入职安全培训才能访问内网。"
        }

    @kernel_function(description="查阅公司政策手册，获取关于特定主题的规定", name="search_policy")
    def search_policy(self, query: str) -> str:
        print(f"    [🔍 RAG 检索] 正在在知识库中搜索关于 '{query}' 的政策...")
        # 简单的关键词匹配模拟向量检索
        if "远程" in query or "WFH" in query or "remote" in query:
            return self.knowledge_base["remote_work"]
        elif "报销" in query or "钱" in query:
            return self.knowledge_base["reimbursement"]
        else:
            return "未在员工手册中找到相关规定。"

# ==============================================================================
# 2. 模拟 HR 数据库 (结构化数据)
# ==============================================================================
class HRDatabasePlugin:
    """
    负责查询员工的个人档案状态
    """
    def __init__(self):
        # 模拟 SQL 数据库
        self.employees = {
            "Alice": {
                "id": "E001",
                "role": "Full-Time",
                "join_date": "2023-01-01", # 入职很久了
                "lateness_count": 0        # 表现良好
            },
            "Bob": {
                "id": "E002",
                "role": "Intern",          # 实习生
                "join_date": "2024-05-01",
                "lateness_count": 2
            }
        }

    @kernel_function(description="根据姓名查询员工的详细档案信息", name="get_employee_info")
    def get_employee_info(self, name: str) -> str:
        print(f"    [🗄️ DB 查询] 正在查询员工 '{name}' 的数据库档案...")
        employee = self.employees.get(name)
        if employee:
            return str(employee)
        return "查无此人"

# ==============================================================================
# 3. 模拟 办公自动化系统 (Action)
# ==============================================================================
class OfficeActionPlugin:
    """
    负责执行具体的审批动作，如发邮件、发通知
    """
    @kernel_function(description="发送审批结果通知邮件", name="send_notification")
    def send_notification(self, employee_name: str, result: str, reason: str) -> str:
        print(f"\n    [📧 发送邮件] TO: {employee_name}@company.com")
        print(f"    [邮件主题] 关于您的远程办公申请结果")
        print(f"    [邮件内容] 结果：{result}。原因：{reason}\n")
        return "邮件发送成功"

# ==============================================================================
# 4. 主程序：构建企业级 Agent
# ==============================================================================
async def main():
    # 初始化 Kernel
    kernel = Kernel()

    # 配置 OpenAI
    service_id = "default"
    kernel.add_service(
        OpenAIChatCompletion(
            service_id=service_id,
            ai_model_id="gpt-4o-mini",
            api_key=os.getenv("OPENAI_API_KEY"),
        )
    )

    # 加载所有插件 (Plugin 组合拳)
    # 1. 给它脑子（知识）
    kernel.add_plugin(CompanyPolicyPlugin(), plugin_name="Policy")
    # 2. 给它眼睛（查数据）
    kernel.add_plugin(HRDatabasePlugin(), plugin_name="HRDB")
    # 3. 给它手（执行）
    kernel.add_plugin(OfficeActionPlugin(), plugin_name="Office")

    # 开启自动工具调用
    settings = OpenAIChatPromptExecutionSettings(
        service_id=service_id,
        function_choice_behavior=FunctionChoiceBehavior.Auto()
    )

    chat_service = kernel.get_service(service_id)
    history = ChatHistory()

    # --- 场景 A: 完美的申请者 Alice ---
    # 这里的难点在于：
    # AI 必须先去 Policy Plugin 查规则，发现规则有 3 条。
    # 然后去 HRDB Plugin 查 Alice 的数据，逐条比对（不是实习生？入职满6个月？没迟到？）。
    # 最后判断通过，调用 Office Plugin 发邮件。

    task = "员工 Alice 申请下周远程办公(WFH)，请根据公司政策和她的个人情况进行审批，并发送通知邮件告知她结果。"

    print(f"🔥 [任务开始]: {task}")
    history.add_user_message(task)

    result = await chat_service.get_chat_message_contents(
        chat_history=history,
        settings=settings,
        kernel=kernel
    )

    print("\n" + "="*50)
    print(f"🤖 [Agent 执行报告]:\n{result[0]}")
    print("="*50)

    # --- 场景 B: 不符合条件的 Bob (可选测试) ---
    # 如果把上面的 Alice 改成 Bob，AI 应该会自动驳回，因为 Bob 是 Intern

if __name__ == "__main__":
    asyncio.run(main())
```

### 运行时的 "思维链" 解析

当你运行这段代码时，请仔细观察控制台输出，你会看到 AI 极其清晰的 **S.O.P. (标准作业程序)**：

1.  **第一步：获取规则 (Information Retrieval)**
    *   AI 意识到它不知道什么是“审批标准”。
    *   调用 `Policy.search_policy("远程办公")`。
    *   **得到知识**：需要入职满6个月 + 无迟到 + 非实习生。

2.  **第二步：获取事实 (Data Gathering)**
    *   AI 意识到它不知道 Alice 符不符合这些标准。
    *   调用 `HRDB.get_employee_info("Alice")`。
    *   **得到数据**：`{role: Full-Time, join_date: 2023...}`。

3.  **第三步：逻辑判别 (Reasoning)**
    *   这是一步隐式的思考（发生在 LLM 大脑里）。
    *   它对比：2023年入职 > 6个月？Yes。
    *   它对比：迟到 0 次 < 1次？Yes。
    *   它对比：Full-Time != Intern？Yes。
    *   结论：**批准 (Approve)**。

4.  **第四步：执行动作 (Execution)**
    *   调用 `Office.send_notification(Alice, "通过", "符合所有政策要求")`。

5.  **第五步：汇报**
    *   向用户返回最终总结。

### 这个案例的价值

这就是目前 **Enterprise Agent (企业级智能体)** 的雏形。

大多数公司的内部系统就是这样：
*   **文档库** (Confluence/SharePoint) -> `CompanyPolicyPlugin`
*   **业务数据库** (ERP/CRM/HRM) -> `HRDatabasePlugin`
*   **办公软件** (Outlook/Teams/Slack) -> `OfficeActionPlugin`

Semantic Kernel 的强大之处在于，它通过统一的 **Plugins** 标准，把这些原本孤立的系统串联起来，让 LLM 充当了那个能够理解规则、查询数据并执行操作的 **"超级中台"**。

既然你已经离开了 Jupyter 环境（例如正在使用 PyCharm, VS Code 或直接在终端运行 `.py` 文件），那么你的代码可以变得更**标准**、更**简洁**了。

在标准的 Python 脚本环境中，你**不再需要** `nest_asyncio`。

### 标准 Python 脚本的写法

在 `.py` 文件中，启动异步程序的标准写法就是直接使用 `asyncio.run(main())`。

下面是**去除所有 Jupyter 兼容补丁**后的、干净的 **HR 合规助手 (HR Compliance Agent)** 完整代码。你可以直接保存为 `.py` 文件运行。

#### 文件名：`hr_agent.py`

```python
import asyncio
import os
import dotenv
from typing import List

# 引入 Semantic Kernel 核心组件
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion, OpenAIChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function

# 加载环境变量 (.env)
dotenv.load_dotenv()

# ==============================================================================
# 1. 模拟向量数据库 (RAG 知识库)
# ==============================================================================
class CompanyPolicyPlugin:
    """
    负责检索公司非结构化的规章制度 (模拟 RAG)
    """
    def __init__(self):
        # 模拟存入向量库的文档切片
        self.knowledge_base = {
            "remote_work": "公司远程办公(WFH)政策：1. 员工必须入职满 6 个月。 2. 过去 30 天内没有迟到早退记录。 3. 实习生(Intern)无权申请。",
            "reimbursement": "报销政策：餐饮报销上限为 50 元/天，交通报销需要发票。",
            "security": "安全政策：所有员工必须完成入职安全培训才能访问内网。"
        }

    @kernel_function(description="查阅公司政策手册，获取关于特定主题的规定", name="search_policy")
    def search_policy(self, query: str) -> str:
        print(f"    [🔍 RAG 检索] 正在在知识库中搜索关于 '{query}' 的政策...")
        if "远程" in query or "WFH" in query or "remote" in query:
            return self.knowledge_base["remote_work"]
        elif "报销" in query or "钱" in query:
            return self.knowledge_base["reimbursement"]
        else:
            return "未在员工手册中找到相关规定。"

# ==============================================================================
# 2. 模拟 HR 数据库 (结构化数据)
# ==============================================================================
class HRDatabasePlugin:
    """
    负责查询员工的个人档案状态
    """
    def __init__(self):
        # 模拟 SQL 数据库
        self.employees = {
            "Alice": {
                "id": "E001",
                "role": "Full-Time",
                "join_date": "2023-01-01",
                "lateness_count": 0
            },
            "Bob": {
                "id": "E002",
                "role": "Intern",
                "join_date": "2024-05-01",
                "lateness_count": 2
            }
        }

    @kernel_function(description="根据姓名查询员工的详细档案信息", name="get_employee_info")
    def get_employee_info(self, name: str) -> str:
        print(f"    [🗄️ DB 查询] 正在查询员工 '{name}' 的数据库档案...")
        employee = self.employees.get(name)
        if employee:
            return str(employee)
        return "查无此人"

# ==============================================================================
# 3. 模拟 办公自动化系统 (Action)
# ==============================================================================
class OfficeActionPlugin:
    """
    负责执行具体的审批动作，如发邮件、发通知
    """
    @kernel_function(description="发送审批结果通知邮件", name="send_notification")
    def send_notification(self, employee_name: str, result: str, reason: str) -> str:
        print(f"\n    [📧 发送邮件] TO: {employee_name}@company.com")
        print(f"    [邮件主题] 关于您的远程办公申请结果")
        print(f"    [邮件内容] 结果：{result}。原因：{reason}\n")
        return "邮件发送成功"

# ==============================================================================
# 4. 主程序
# ==============================================================================
async def main():
    print(">>> 系统启动中...")

    # 初始化 Kernel
    kernel = Kernel()

    # 配置 OpenAI 服务
    service_id = "default"
    try:
        kernel.add_service(
            OpenAIChatCompletion(
                service_id=service_id,
                ai_model_id="gpt-4o-mini",
                api_key=os.getenv("OPENAI_API_KEY"),
            )
        )
    except Exception as e:
        print(f"OpenAI 服务初始化失败: {e}")
        return

    # 加载插件
    kernel.add_plugin(CompanyPolicyPlugin(), plugin_name="Policy")
    kernel.add_plugin(HRDatabasePlugin(), plugin_name="HRDB")
    kernel.add_plugin(OfficeActionPlugin(), plugin_name="Office")

    # 配置自动工具调用 (SK v1.x 标准写法)
    settings = OpenAIChatPromptExecutionSettings(
        service_id=service_id,
        function_choice_behavior=FunctionChoiceBehavior.Auto()
    )

    chat_service = kernel.get_service(service_id)
    history = ChatHistory()

    # --- 执行任务 ---
    task = "员工 Alice 申请下周远程办公(WFH)，请根据公司政策和她的个人情况进行审批，并发送通知邮件告知她结果。"
    print(f"🔥 [接收任务]: {task}\n")
    history.add_user_message(task)

    try:
        # 获取回复
        result = await chat_service.get_chat_message_contents(
            chat_history=history,
            settings=settings,
            kernel=kernel
        )

        print("="*50)
        print(f"🤖 [Agent 最终反馈]:\n{result[0]}")
        print("="*50)

    except Exception as e:
        print(f"运行时发生错误: {e}")

# ==============================================================================
# 5. 程序入口 (标准写法)
# ==============================================================================
if __name__ == "__main__":
    # 在非 Jupyter 环境下，直接运行 asyncio.run 即可
    # 不需要 nest_asyncio
    asyncio.run(main())
```

### 运行方式

1.  确保你的目录下有 `.env` 文件，且包含 `OPENAI_API_KEY`。
2.  在终端（Terminal）中运行：
    ```bash
    python hr_agent.py
    ```

### 为什么这样写是“标准”的？

1.  **`if __name__ == "__main__":`**: 确保这个文件被当作脚本运行时才执行，而被别的模块 import 时不执行。
2.  **`asyncio.run(main())`**: 这是 Python 3.7+ 官方推荐的启动异步程序的入口。它会自动创建一个新的 Event Loop，运行 `main()` 协程，并在结束后自动关闭 Loop，清理资源。这比在 Jupyter 里手动处理 Loop 要安全和干净得多。